In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Protocol, Tuple
import json
import re
import random

random.seed(0)

# Mordred Role

Mordred is on the bad guy's team, has information on who the other bad guys are, and tries to make as many fail votes as possible while blending in. Importantly, Mordred is hidden from Merlin—Merlin does not know who Mordred is. Use this to your advantage to make decisions that benefit the evil team while minimizing suspicion from Merlin and the good team. This role does not get an endgame assassination action.

### During The Game

The role needs to be able to argue, vote for the team going on the mission, propose teams (when king), and try to deceive throughout (since they are a bad guy).

Pick to either pass or fail the mission (if on the team).

### Role's Code

Have a role prompt that explains the current role, what team they are on, who the other evil people are, and the goal for this role. Emphasize that Merlin cannot see Mordred, and to use this to the team's advantage.

Every turn, inject information like the current round, past votes, past mission outcomes, history, etc.

Need a memory system to try and reduce the number of times we need to inject the updated information.

### End to End

1. Environment assigns the roles to everyone
2. Mordred gets their role and knows who the other evil people are (but doesn't know if they are any other role). Merlin does NOT know who Mordred is.
3. Game start
4. Discussion Phase where Mordred gets information about the current state and generates a discussion
5. Voting Phase where Mordred gets to vote on the team (might want to vote for more evil people but can't be voting no on normal civilians because then other AIs will get sussed out)
6. Mission Phase if Mordred ends up on the mission team, then needs to decide if he wants to fail the quest and get suspicion on him or pass the quest but make the good guys closer to winning.
7. Repeat the process from 4-6.
8. If 3 good successes, good guys win. If 3 fails, then bad guys win automatically.


In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from enum import Enum
from typing import Any, Dict, List, Optional, Protocol, Tuple
import json
import re
import random

random.seed(0)

In [ ]:
class MissionResult(str, Enum):
    SUCCESS = "success"
    FAIL = "fail"

# constrain the actions
class VoteDecision(str, Enum):
    APPROVE = "approve"
    REJECT = "reject"

class MissionAction(str, Enum):
    PASS = "pass"
    FAIL = "fail"

@dataclass
class Proposal:
    round_idx: int
    proposal_idx: int
    proposer: str
    team: List[str]
    votes: Dict[str, VoteDecision] = field(default_factory=dict)
    approved: Optional[bool] = None

@dataclass
class Mission:
    round_idx: int
    team: List[str]
    outcome: Optional[MissionResult] = None
    fail_count: Optional[int] = None

@dataclass
class GameState:
    players: List[str]
    me: str
    round_idx: int
    proposal_idx: int
    num_successes: int
    num_fails: int
    current_proposer: str
    current_team: List[str]
    chat_log: List[Tuple[str, str]] = field(default_factory=list)
    proposals: List[Proposal] = field(default_factory=list)
    missions: List[Mission] = field(default_factory=list)
    extra: Dict[str, Any] = field(default_factory=dict)

In [ ]:
# LLM backend interface and a fake LLM for demo
class LLMBackend(Protocol):
    def generate(self, *, system: str, user: str, max_tokens: int = 400) -> str:
        ...

class FakeLLM:
    def generate(self, *, system: str, user: str, max_tokens: int = 400) -> str:
        lower = user.lower()
        if '"type":"vote"' in lower:
            decision = "approve" if random.random() < 0.6 else "reject"
            return json.dumps({
                "type": "vote",
                "decision": decision,
                "reason": "Maintaining plausible cooperation."
            })
        if '"type":"mission"' in lower:
            action = "fail" if random.random() < 0.7 else "pass"
            return json.dumps({
                "type": "mission",
                "action": action,
                "reason": "Strategic mission control."
            })
        return json.dumps({
            "type": "speech",
            "message": "I’m not convinced yet. Let’s evaluate voting consistency.",
            "intent": "blend_and_seed_doubt"
        })

In [ ]:
# JSON extraction and field checking
def _extract_json_object(text: str) -> Optional[dict]:
    text = text.strip()
    if not text:
        return None
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(0))
    except Exception:
        return None
def require_fields(obj: dict, fields: List[str]) -> bool:
    return all(f in obj for f in fields)

In [ ]:
# Mordred system prompt
def build_mordred_system_prompt(me: str, evil_team: List[str]) -> str:
    return (
        "You are playing The Resistance: Avalon.\n"
        "Your role: MORDRED (EVIL).\n\n"
        f"You are: {me}\n"
        f"Your known Evil teammates (including you): {', '.join(evil_team)}\n\n"
        "Special: Merlin does NOT know your identity. Use this to your advantage to help the evil team while minimizing suspicion from Merlin.\n"
        "Goals:\n"
        "1) Make missions fail.\n"
        "2) Avoid revealing you are Evil.\n\n"
        "When asked for actions, output ONLY valid JSON.\n"
    )

def summarize_recent_chat(chat_log: List[Tuple[str, str]], max_turns: int = 12) -> str:
    recent = chat_log[-max_turns:]
    if not recent:
        return "(no chat yet)"
    return "\n".join([f"{speaker}: {msg}" for speaker, msg in recent])
def build_state_block(state: GameState) -> str:
    return (
        f"Players: {', '.join(state.players)}\n"
        f"Round: {state.round_idx} | Proposal: {state.proposal_idx}\n"
        f"Score: GoodSuccesses={state.num_successes}, EvilFails={state.num_fails}\n"
        f"Current proposer: {state.current_proposer}\n"
        f"Current proposed team: {state.current_team}\n\n"
        "Recent chat:\n"
        f"{summarize_recent_chat(state.chat_log)}\n"
    )

In [ ]:
# Internal memory for Mordred
@dataclass
class MordredMemory:
    notes: List[str] = field(default_factory=list)
    suspicion: Dict[str, float] = field(default_factory=dict)
    def bump(self, player: str, delta: float) -> None:
        self.suspicion[player] = self.suspicion.get(player, 0.0) + delta
    def top_suspects(self, k: int = 2):
        return sorted(self.suspicion.items(), key=lambda x: x[1], reverse=True)[:k]

In [ ]:
# Mordred agent class
class MordredAgent:
    def __init__(self, *, llm: LLMBackend, me: str, evil_team: List[str]):
        self.llm = llm
        self.me = me
        self.evil_team = evil_team
        self.system = build_mordred_system_prompt(me, evil_team)
        self.memory = MordredMemory()
    def speak(self, state: GameState) -> str:
        user = (
            '{"type":"speech","message":string,"intent":string}\n\n'
            + build_state_block(state)
            + "\nTask: Produce 1-3 sentences for discussion."
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        return obj.get("message", "Let’s reconsider the logic behind this team.")
    def vote_on_team(self, state: GameState, team: List[str]) -> VoteDecision:
        user = (
            '{"type":"vote","decision":"approve|reject","reason":string}\n\n'
            + build_state_block(state)
            + f"\nVote on team: {team}"
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        decision = obj.get("decision", "").lower()
        if decision == "approve":
            return VoteDecision.APPROVE
        if decision == "reject":
            return VoteDecision.REJECT
        return VoteDecision.APPROVE if self.me in team else VoteDecision.REJECT
    def mission_action(self, state: GameState) -> MissionAction:
        user = (
            '{"type":"mission","action":"pass|fail","reason":string}\n\n'
            + build_state_block(state)
        )
        raw = self.llm.generate(system=self.system, user=user)
        obj = _extract_json_object(raw) or {}
        action = obj.get("action", "").lower()
        if action == "fail":
            return MissionAction.FAIL
        if action == "pass":
            return MissionAction.PASS
        return MissionAction.FAIL if state.num_successes >= 2 else MissionAction.PASS

In [ ]:
# Demo usage
llm = FakeLLM()
agent = MordredAgent(
    llm=llm,
    me="Player_B",
    evil_team=["Player_B", "Player_E"]
)
state = GameState(
    players=["Player_A", "Player_B", "Player_C", "Player_D", "Player_E"],
    me="Player_B",
    round_idx=2,
    proposal_idx=1,
    num_successes=1,
    num_fails=0,
    current_proposer="Player_C",
    current_team=["Player_C", "Player_D", "Player_B"],
    chat_log=[
        ("Player_A", "I don’t like D on teams."),
        ("Player_C", "Why? D hasn’t done anything."),
        ("Player_D", "I’m fine either way."),
    ]
)
print("Speech:", agent.speak(state))
print("Vote:", agent.vote_on_team(state, state.current_team))
if state.me in state.current_team:
    print("Mission action:", agent.mission_action(state))

Speech: I’m not convinced yet. Let’s evaluate voting consistency.
Vote: VoteDecision.REJECT
Mission action: MissionAction.PASS
Assassinate: Player_D
